# MOM — Risk-Factor Construction Spec (USE4-faithful)

**Purpose.** This notebook is the **source-of-truth spec** for building the **MOM (Momentum)** style factor exactly as MSCI describes it in *The Barra US Equity Model (USE4) — Empirical Notes, September 2011*. It is **not** a research notebook. The deliverable is a clean, USE4-faithful Momentum factor written to parquet, suitable for input to a multi-factor risk model.

**Audience.** You — plus whatever AI assistant you are driving. Treat its output as deeply untrustworthy until audited. Follow the stages linearly. Each stage has:
- ✅ **PDF SPEC** — exact verbatim quote or paraphrase from the USE4 documentation
- ❓ **NOT IN PDF** — something the PDF does not specify; a judgment call you must make
- 💡 **DEFAULT** — a reasonable default for any ❓ item, chosen to match standard Barra practice
- 🧪 **VALIDATE** — sanity checks to run after each stage

**Inputs:** Daily prices from `data/cleaned/sharadar/prices/` (adjusted close, market cap). Ken French daily risk-free rate. No fundamentals. No upstream factor outputs.

**Deliverable:** A parquet file `mom_use4.parquet` keyed by `(permaticker, signal_date)` containing the standardised Momentum exposure for every stock in the coverage universe on every monthly signal date.

**Companion specs:**
- `02_style_factors/beta/beta_spec.ipynb` — Beta shares the exponential-decay weighting scheme and excess-return infrastructure. MOM is structurally simpler: a weighted sum rather than an OLS slope.
- `02_style_factors/nlb/nlb_spec.ipynb` — shares the standardisation and outlier-trim machinery.

---

### Critical reading

The USE4 PDF specifies MOM in **two short places**:
- **Section 3.4 (Style Factors, p.16–17)** — qualitative description
- **Appendix A (p.51)** — formal descriptor definition

Both are quoted verbatim in §1 below. Everything else in this spec is either derived from elsewhere in the PDF (standardisation convention, outlier trim) or marked ❓ NOT IN PDF.

**Bias toward the literal spec.** Where the PDF is ambiguous, follow the conventions established in `beta_spec.ipynb` and `nlb_spec.ipynb` — MOM shares their excess-return infrastructure and standardisation pipeline.

## §1. The USE4 MOM spec — verbatim quotes

### 1a. Empirical Notes, Appendix A (p.51) — formal descriptor definition

> **Momentum**
>
> *Definition:* `1.0 · RSTR`
>
> *RSTR — Relative Strength:* *"Cumulative 504-day excess log return over the 504 trading days ending 21 trading days before the current date. Exponential weight with half-life of 126 days."*

### 1b. Appendix A (p.51) — formula

$$\text{mom}_{i,t_0} = \sum_{d=L+1}^{T+L} w_d \cdot \left[\ln(1+r_{i,d}) - \ln(1+r_{f,d})\right]$$

where:

| Symbol | Value | Description |
|---|---|---|
| $r_{i,d}$ | — | Adjusted daily log return of stock $i$, observed $d$ trading days before $t_0$ |
| $r_{f,d}$ | — | Daily risk-free rate, $d$ trading days before $t_0$ |
| $w_d$ | $\exp(-\ln 2 \cdot d / 126)$ | Exponential decay weight; **higher weight for smaller $d$** (more recent) |
| $T$ | 504 trading days | Return window length (~2 years) |
| $L$ | 21 trading days | Skip lag (~1 month, to avoid short-term reversal) |
| $HL$ | 126 trading days | Weight half-life (~6 months) |

So the estimation window spans **d = 22 through d = 525** trading days before the signal date (504 observations maximum). The most recent 21 trading days are excluded.

### 1c. Empirical Notes, Table 4.2 (p.26) — reported factor statistics

USE4 reports for MOM:
- **Average |t|:** ~3.5 (one of the stronger style factors)
- **Factor stability:** ~0.89–0.92 (moderate; momentum mean-reverts over multi-year horizons)
- **Variance Inflation Factor:** low (MOM is relatively orthogonal to other style factors)

### 1d. Methodology Notes, Section 2.3 (p.9) — standardisation rule (applies to all style factors)

> *"$d_{nl} = (d_{nl}^{\text{Raw}} - \mu_l) / \sigma_l$ where $\mu$ is cap-weighted mean and $\sigma$ is equal-weighted std within ESTU."*

### 1e. Methodology Notes, Section 2.2 (p.8) — outlier treatment (applies to all descriptors)

> *"We trim these observations to three standard deviations from the mean."*

---

**That is all the PDF says about MOM construction.** The formula, window (T=504), lag (L=21), and half-life (HL=126) are fully specified. The remaining implementation decisions — risk-free rate source, weight normalisation, minimum observations threshold, calendar start — are ❓ **NOT IN PDF** and are flagged in the stages below.

## §2. End-to-end pipeline

```
┌────────────────────────────────────────────────────────────────────┐
│  UPSTREAM:  estu_build.ipynb   →  estu_monthly.parquet             │
│             daily_panel_build.ipynb  →  daily_returns.parquet (+ maps)   │
├────────────────────────────────────────────────────────────────────┤
│  STAGE 0:  Setup, parameters                                       │
│  STAGE 1:  Load shared daily-returns artifact                      │
│  STAGE 2:  Load shared ESTU                                        │
│  STAGE 3:  _td_to_sig map (benchmark already in daily.mkt_ret)     │
│  STAGE 4:  MOM estimator — 504d window, 21d lag, 126d half-life    │
│  STAGE 5:  Outlier trim (3σ)                                       │
│  STAGE 6:  Standardise (CW mean = 0, EW std = 1)                   │
│  STAGE 7:  Save deliverable                                        │
│  STAGE 8:  Validate                                                │
└────────────────────────────────────────────────────────────────────┘
```

**PIT discipline:** for any signal_date `t`, only data dated ≤ `t` is used.

**One-sweep run:** `your end-to-end runner` executes the whole
pipeline in dependency order (sequential, one kernel at a time).

## §3. Output schema — what the worker delivers

**Raw descriptor column:** `mom`
**Standardised score column:** `mom_score`

**File:** `data/out/mom_use4.parquet`

**Compression:** zstd, statistics=True

**Schema:**

| Column | Type | Description |
|---|---|---|
| `permaticker` | Int64 | Sharadar permanent ticker ID |
| `signal_date` | Date | End-of-month rebalance date (last trading day of month) |
| `in_estu` | Bool | Whether this stock was in the estimation universe on this date |
| `mcap` | Float64 | Market cap on signal_date (used for cap-weighting) |
| `mom` | Float64 | **Raw descriptor** — normalised exponentially weighted excess log return (weighted-average daily excess return) |
| `mom_score` | Float64 | **Final MOM exposure** — outlier-trimmed and standardised cross-sectionally (the deliverable) |
| `n_obs` | UInt32 | Number of valid excess-return observations in the [L+1, T+L] estimation window |

**Coverage rules:**
- Compute `mom` and `mom_score` for **every stock with `n_obs ≥ MIN_OBS=252`** in the window, not just ESTU members.
- Standardisation moments (mean, std) are computed using **ESTU members only**.
- Non-ESTU stocks get the same standardisation parameters applied so the final factor is comparable across the coverage universe.
- A stock with `n_obs < MIN_OBS` receives `null` for `mom` and `mom_score`; its `n_obs` is still recorded.

## §4. STAGE 0 — Setup, imports, paths

Standard imports. Use polars, not pandas, throughout (project convention). The `bisect` module is needed for PIT trading-day lookups (same as Beta).

✅ **PDF SPEC — parameters explicitly defined (Appendix A, p.51):**

> `T = 504` trading days — *"Cumulative 504-day excess log return"*
>
> `L = 21` trading days — *"ending 21 trading days before the current date"*
>
> `HL = 126` trading days — *"Exponential weight with half-life of 126 days"*

❓ **NOT IN PDF — minimum observations threshold.** The spec does not define when a stock has "sufficient" history.

💡 **DEFAULT:** `MIN_OBS = 252` (~1 year). A stock must have at least 252 valid excess-return observations within the [L+1, T+L] window. This filters out recent IPOs and ensures the weighted sum is not dominated by a handful of observations near the beginning of the stock's history.

❓ **NOT IN PDF — calendar start.**

💡 **DEFAULT:** `CALENDAR_START = date(1999, 1, 1)`. Match Beta and NLB for consistency. Note: the first signal dates (1999) will have thin histories for most stocks since MIN_OBS=252 requires ~1 year of trading history inside the window. First dates with broad coverage (~4,000+ qualifying stocks) will be around 2001.

```python
# ───── Parameters ─────────────────────────────────────────────────────────────

# ✅ PDF SPEC — Appendix A p.51
MOM_WINDOW_DAYS  = 504   # T: return window length in trading days
MOM_HALF_LIFE_TD = 126   # HL: exponential weight half-life in trading days
MOM_LAG_DAYS     = 21    # L: skip lag in trading days (excludes short-term reversal)

# 💡 DEFAULT (NOT IN PDF)
MIN_OBS        = 252            # minimum valid observations required for a non-null score
CALENDAR_START = date(1999, 1, 1)

# -- Factor type (read by /build-factor to select boilerplate template)
FACTOR_TYPE = "time_series"   # time_series or fundamental
```

## §5. STAGE 1 — Load the shared daily panel

By now you have built this plumbing inline at least once (Beta). This factor
consumes the **shared daily panel** instead — the refactor step specified in
`daily_panel/daily_panel_spec.ipynb`:

| Variable | File (in `data/out/`) | Contents |
|---|---|---|
| `daily` | `daily_returns.parquet` | `permaticker, date, ret, mcap_lag, mkt_ret` — excess log returns; `mkt_ret` is the ESTU cap-weighted excess benchmark |
| `prices` | `ticker_permaticker.parquet` | ticker → permaticker map (spot-check lookups) |

**Build order:** `estu_build.ipynb` → `daily_panel_build.ipynb` → this factor.

🧪 **VALIDATE:** performed once in the daily-panel build — benchmark correlation
> 0.90, crisis-vol sanity, load-path equivalence, missing-benchmark dates < 30.

## §6. STAGE 2 — Load the shared ESTU (`estu_monthly.parquet`)

✅ **PDF SPEC:** USE4 ESTU = MSCI USA Investable Markets Index (USA IMI) — ~99% of
US float-adjusted market cap, liquidity- and stability-screened, ~2,500–3,000 names.

💡 **DEFAULT:** load the shared `data/out/estu_monthly.parquet` built by
`estu_build.ipynb` (spec: `01_estu/estu_spec.ipynb`): top ~3,000
domestic common stocks on NYSE/NASDAQ/NYSEMKT, ATVR liquidity screen
(add ≥ 20%, retain ≥ 10%), 3,000/3,500 hysteresis buffer. **Every factor must use
this same ESTU** — factor-specific universes would break the multi-factor
cross-sectional regression.

🧪 **VALIDATE:**
- 330 monthly signal dates (1999-01-29 →); ESTU size mean ≈ 2,500, range ≈ 1,800–3,050
- Mega-caps (AAPL, MSFT, GOOGL, JPM) present every month they were public

## §7. STAGE 3 — ESTU benchmark (already in the daily artifact)

✅ **PDF SPEC:** factor regressions use the cap-weighted estimation-universe return.

💡 **DEFAULT:** `daily.mkt_ret` already carries the ESTU cap-weighted **excess**
benchmark, built and validated once in `daily_panel_build.ipynb`. This stage only derives
`_td_to_sig` (trading day → owning signal date, via `factor_lib.make_td_to_sig`)
for the validation quintile checks.

🧪 **VALIDATE:** missing `mkt_ret` dates after the first signal date < 30.

## §8. STAGE 4 — MOM estimator

✅ **PDF SPEC (Empirical Notes, Appendix A, p.51):**

> *"RSTR — Relative Strength: Cumulative 504-day excess log return over the 504 trading days ending 21 trading days before the current date. Exponential weight with half-life of 126 days."*

**Formula:**

```
For each stock i and signal_date t_0:

    window_days = [d for d in trading_days if L < (t_0_idx - d_idx) <= T + L]
               = trading days at positions [i_t0 - (T+L), i_t0 - L)  in the sorted calendar
               = days at lags d = 22, 23, ..., 525  (504 days total, maximum)

    w_d         = exp(−ln(2) · d / HL)          # d = lag in trading days; smaller d → higher weight
    exc_ret_d   = ln(1 + r_{i,d}) − ln(1 + r_{f,d})

    mom_i       = Σ(w_d · exc_ret_d) / Σ(w_d)  # normalised: weighted-average daily excess return
    n_obs_i     = count of valid (non-null) exc_ret_d in window

    If n_obs_i < MIN_OBS:  mom_i = null
```

**Window boundaries** (confirmed):
- Most recent observation: **d=22** (22 trading days before signal_date; the day immediately after the 21-day skip)
- Oldest observation: **d=525** (525 trading days before signal_date; 504 days after the skip boundary)
- The 21 days d=1..21 are **excluded** from the window (short-term reversal buffer)

**Weights at the boundaries:**
- `w_22  = exp(−ln(2)·22/126)  ≈ 0.887`  (most recent, highest weight in window)
- `w_525 = exp(−ln(2)·525/126) ≈ 0.055`  (oldest, lowest weight)

❓ **NOT IN PDF — weight normalisation.** The USE4 formula is a weighted sum with no explicit denominator. If n_obs < T=504 (IPO-stage stocks, thin Sharadar coverage), the unnormalised sum is smaller in absolute value even for equally strong momentum, creating a cross-sectional bias toward zero.

💡 **DEFAULT:** Normalise by dividing by `Σw_d` (sum of active weights). This converts `mom` from a cumulative quantity to a **weighted-average daily excess return**, comparable across stocks regardless of history length. The cross-sectional standardisation in Stage 6 absorbs any scale differences, but normalisation makes the raw `mom` column interpretable and eliminates the n_obs bias.

❓ **NOT IN PDF — treatment of missing daily returns within the window.** Some trading days may have null excess returns (stock halted, thin Sharadar coverage, first available row for a stock).

💡 **DEFAULT:** Skip nulls — include only days with valid excess returns in both the numerator and denominator. The `n_obs` column records how many valid observations contributed. A day with a null return contributes zero weight to `Σw_d` (i.e., it is excluded from both numerator and denominator).

❓ **NOT IN PDF — trading-day calendar.** What constitutes a "trading day" for counting the lag L and window T?

💡 **DEFAULT:** Use the master trading-day list derived from the full price panel (all `date` values present in `prices/*.parquet`), consistent with Beta. Use `bisect.bisect_right` on this sorted list to locate `i_t0` (the index of signal_date), then slice `[i_t0 - (T+L) : i_t0 - L]` to get exactly 504 window dates.

---
*The section above (PDF SPEC quote through final 💡 DEFAULT) is the `STAGE4_DESCRIPTION` that `/build-factor` will inject verbatim into the Stage 4 stub docstring. Content below this line is supplementary guidance for the implementer and is not extracted.*
---

**Implementation approach — two options:**

**Option A — vectorized (recommended if ≥32 GB RAM):**
1. Pre-compute `d = i_t0 - i_date` for every `(permaticker, date, signal_date)` triple using a broadcast join
2. Filter to `d ∈ [L+1, T+L]`; compute `w_d = exp(-ln(2) * d / HL)`
3. `group_by(["permaticker", "signal_date"]).agg(Σ(w·exc_ret)/Σw, n_obs=count)`
4. Memory note: the full window join expands 39M daily rows by up to 504× — use lazy evaluation and streaming if needed

**Option B — per-signal-date loop (safe on low RAM):**
```python
trading_days = sorted(...)   # master list of all market open days
results = []
for sd in tqdm(monthly_cal):
    i_sd   = bisect.bisect_right(trading_days, sd) - 1
    win    = set(trading_days[max(0, i_sd - (T+L)) : i_sd - L])
    chunk  = daily_panel.filter(pl.col("date").is_in(win))
    # for each permaticker in chunk: compute lag d, weight, weighted sum
    chunk  = (chunk
              .with_columns(
                  pl.lit(i_sd).alias("i_sd"),
                  # map date to index, compute d = i_sd - i_date
              )
              .group_by("permaticker")
              .agg(mom=..., n_obs=...))
    chunk  = chunk.with_columns(pl.lit(sd).cast(pl.Date).alias("signal_date"))
    results.append(chunk)
panel = pl.concat(results)
```

**`_FAST` flag:** Set `_FAST = False` on Jakob's machine (≤16 GB available RAM). Option B processes one signal_date at a time and keeps memory usage O(one-month daily panel).

🧪 **VALIDATE (immediately after Stage 4, before trimming):**
- For a stock with full 504-day history: `n_obs = 504`, `mom` is in the plausible range ~[−0.003, +0.003] per day
- Cross-sectional distribution of `mom` is roughly bell-shaped (not skewed like NLS)
- High-momentum tech names (post-2013 period): AAPL, AMZN, NVDA should have positive `mom`
- Defensive names during bull markets (KO, JNJ in 2013–2014): should have negative or near-zero `mom`
- `n_obs` is never > T=504 and never null for rows where `mom` is non-null

## §9. STAGE 5 — Outlier trim (3σ)

✅ **PDF SPEC (Methodology Notes §2.2, p.8):**

> *"We trim these observations to three standard deviations from the mean."*

This general rule applies to all USE4 descriptors. For MOM specifically: the raw momentum descriptor is approximately bell-shaped (unlike NLS's heavily skewed cubic), so σ-based trimming is well-suited and will clip a small fraction of the distribution symmetrically.

**Per signal_date, using only ESTU members to compute trim bounds, applied to all stocks:**

```
μ_trim = Σ_{i ∈ ESTU} (mcap_i · mom_i) / Σ mcap_i   # cap-weighted mean
σ_trim = std_{i ∈ ESTU}(mom_i)                         # equal-weighted std
lo = μ_trim − 3 · σ_trim
hi = μ_trim + 3 · σ_trim
mom_trimmed_i = clip(mom_i, lo, hi)    ∀ i in coverage universe with non-null mom
```

❓ **NOT IN PDF — which mean for the trim bounds.** "Three standard deviations from the mean" could use equal-weighted or cap-weighted mean.

💡 **DEFAULT:** Cap-weighted mean for the centre, equal-weighted std for the width — consistent with the standardisation step (Stage 6) and with Beta's trim implementation. This means trim bounds and standardisation bounds use the same anchor, making the two steps coherent.

🧪 **VALIDATE:**
- ~0.5–2% of ESTU rows clipped per date (momentum is roughly symmetric; expect roughly equal clipping at both tails)
- After trimming, cross-sectional skewness of `mom_trimmed` should be lower than `mom` in periods with momentum crashes (2001, 2009)

## §10. STAGE 6 — Standardise (cap-weighted mean=0, EW std=1)

✅ **PDF SPEC (Methodology Notes §2.3, p.9):**

> *"μ_l is the cap-weighted mean of the descriptor (within the estimation universe), and σ_l is the equal-weighted standard deviation."*

**Per signal_date `t`, using only ESTU members:**

```
μ_t  = Σ_{i ∈ ESTU_t} (mcap_i · mom_trimmed_i) / Σ mcap_i   # cap-weighted mean
σ_t  = std_{i ∈ ESTU_t}(mom_trimmed_i)                        # equal-weighted std
mom_score_{i,t} = (mom_trimmed_i − μ_t) / σ_t                 # applied to ALL i
```

Three critical points:
1. Moments estimated on **ESTU only**; applied to **every stock** in the coverage universe.
2. Mean is **cap-weighted**; std is **equal-weighted**. Do not homogenise — the asymmetry is intentional (§2.3).
3. Result: the cap-weighted ESTU portfolio has exactly zero MOM exposure.

🧪 **VALIDATE:**
- `Σ_{i ∈ ESTU} (mcap_i · mom_score_i) / Σ mcap_i ≈ 0` (cap-weighted mean = 0 by construction; |mean| < 1e-6)
- `std_{i ∈ ESTU}(mom_score_i) ≈ 1.0` (equal-weighted std = 1 by construction; |std − 1| < 0.02)

## §11. STAGE 7 — Save deliverable

Write the final panel to `data/out/mom_use4.parquet`.
Compression: zstd. Statistics: True.

Include `mom` (raw trimmed descriptor) and `n_obs` for downstream audit. The downstream consumer uses `mom_score`.

Verify on write:
- Schema matches §3 exactly (column names, dtypes)
- Row count and date range are as expected
- File size is reasonable (expect 40–60 MB, similar to NLS)

## §12. STAGE 8 — Validation

Run these checks **on the saved deliverable**, not on intermediate state.

### Required checks (all must pass before signing off)

**1. Standardisation moments (ESTU):**
```
For each signal_date t, computed over ESTU members with non-null mom_score:
    |cap-weighted mean(mom_score)| < 1e-6
    |equal-weighted std(mom_score) − 1| < 0.02
```

**2. Raw descriptor scale:**
```
The normalised mom descriptor is a weighted-average daily excess return.
For the post-2001 sample, expected range of ESTU median mom ≈ [−0.0005, +0.0005] per day
(roughly ±12% annualised in the typical cross-section).
Extreme values (|mom| > 0.003) should be rare after trimming.
```

**3. Coverage stability:**
```
Count of stocks with non-null mom_score per signal_date should be stable post-2001.
Target: ≥ 4,000 stocks per date post-2005.
Note: 1999–2000 dates will have far fewer qualifying stocks (MIN_OBS=252 requires
~1 year inside the window; first dates with broad coverage ≈ 2001).
Plot coverage over time — a sudden drop suggests a data-loading bug.
```

**4. Factor stability (MoM Spearman ρ, ESTU only):**
```
For consecutive signal_dates t and t+1:
    Spearman ρ of mom_score for stocks present in ESTU on both dates
Target: mean ρ > 0.85
Rationale: each new month drops the oldest 21 days and adds 21 new ones
(≈4% window turnover), so ranks change slowly. USE4 Table 4.2 reports ~0.89–0.92.
Restrict to ESTU for consistency with NLB/NLS checks.
```

**5. Economic direction (Q5-Q1 spread vs ESTU market return):**
```
Construct equal-weight Q5 and Q1 portfolios from ESTU mom_score quintiles.
Compute daily Q5-Q1 spread return.
Pearson corr(Q5-Q1 daily spread, ESTU mkt_ret) should be > 0.10.
Note: correlation will be modest — MOM is NOT a market beta proxy.
Momentum crashes (e.g., March–June 2009) will produce large negative spread days
that temporarily drive this correlation negative; this is empirically documented
and expected, not a bug.
```

**6. Disk vs memory equivalence:**
```
At one spot-check signal_date: max abs diff of mom_score values < 1e-10.
```

### Optional diagnostic checks

- Time series of cross-sectional median `mom` (ESTU): should fluctuate near zero with no persistent drift. Momentum crashes visible as sharp negative spikes (2001, 2009).
- Histogram of `mom_score` per date: roughly bell-shaped, centred at zero, range mostly [−3, +3].
- Spot check 2013-12-31: AMZN, GOOG, NFLX should have high positive `mom_score`; defensive stocks (KO, SO) should have negative `mom_score`.
- Verify `n_obs ≥ MIN_OBS=252` for every row with non-null `mom_score`; no exceptions.

---

**Shared validation conventions (all factors, 2026-06-10):**
- **Coverage (check 3)** is evaluated on **completed months only** — the final
  signal date is the in-progress month (freshest data can lag the Sharadar
  refresh) and is excluded from the floor.
- **Fraction of scores in [−3, +3]** is reported for the full universe *and* for
  ESTU; the ESTU figure is the operationally relevant one (non-ESTU micro-caps
  pull the all-universe tail).
- Common check machinery lives in `common/ (your shared utilities)`
  .


## §13. Master list of ❓ NOT-IN-PDF decisions

| # | Decision | Default chosen | Alternative | Where to revisit |
|---|---|---|---|---|
| 1 | Minimum observations threshold | 252 (~1 year) | 126 (permissive), 504 (strict, full-window only) | If early-date coverage is too thin or noisy |
| 2 | Weight normalisation | Normalise by Σw (weighted-average daily return) | Raw sum per USE4 literal (no denominator) | If raw cumulative sum is preferred for research comparisons |
| 3 | Risk-free rate source | Ken French daily RF (F-F Data Library) | 3-month T-bill, Fed Funds overnight | Rarely worth changing; Ken French is the standard for academic USE4 replications |
| 4 | Treatment of null daily returns in window | Skip nulls; exclude from both numerator and denominator | Fill nulls with 0 (assume flat) or with the day's RF | If Sharadar coverage gaps are systematic for specific names |
| 5 | Calendar start | 1999-01-01 (match Beta/NLB) | 2001-01-01 (first dates with ~4,000 qualifying stocks) | If 1999–2000 thin-coverage dates cause downstream issues |
| 6 | Trim bound convention | CW mean ± 3×EW std (ESTU) | EW mean ± 3×EW std | Consistent with standardisation; almost no empirical difference |
| 7 | Trading-day calendar | Derived from master price panel (all dates in prices/*.parquet) | External exchange calendar (pandas_market_calendars) | If Sharadar has gaps that cause calendar artifacts |
| 8 | Adjusted price field | closeadj | closeunadj (wrong), fcloseadj if available | Never use closeunadj for momentum; splits create false signals |

## §14. Common pitfalls — read this before coding

**Pitfall 1: Off-by-one in the skip-lag window.**
The L=21 skip means the window starts at d=L+1=22, NOT d=L=21. Including d=21 contaminates the descriptor with short-term reversal. Verify: the most recent observation in the window is the close price exactly **22** trading days before signal_date, not 21. Similarly, the oldest observation is at d=T+L=525, not T+L−1=524 — a symmetric off-by-one at the far end.

**Pitfall 2: Calendar days vs trading days.**
L=21 and T=504 are in **trading days**, not calendar days. 21 trading days ≈ 1 calendar month; 504 trading days ≈ 2 calendar years. Never compute the window boundary as `signal_date - timedelta(days=21*7/5)`. Always index backward in the sorted trading-day list by position count.

**Pitfall 3: Subtracting RF at the wrong point.**
Subtract the **daily** risk-free rate from each daily log return **before** accumulating into the weighted sum. Do NOT compute the raw weighted cumulative log return first and then subtract a single RF quantity at the end — the exponential weights make this incorrect (the RF subtracted per day interacts with the weighting in a way that only cancels properly when done per observation).

**Pitfall 4: Using `closeunadj` instead of `closeadj`.**
Momentum MUST use split- and dividend-adjusted prices. `closeunadj` produces large artificial negative returns on ex-dividend dates (especially high-yield stocks) and step-discontinuities at stock splits, both of which contaminate the momentum signal systematically.

**Pitfall 5: Unnormalised weights create a bias toward zero for thin-history stocks.**
Without the `Σw_d` denominator, a stock with 252 valid window days has a raw weighted sum roughly half as large as a stock with 504 days of equal momentum strength. After cross-sectional standardisation, thin-history stocks are systematically pulled toward the median. The normalised weighted average eliminates this bias.

**Pitfall 6: Filtering delisted stocks from the daily panel before building the window.**
A stock that delisted before signal_date but was active during the estimation window should still contribute its historical returns to other stocks' windows (if using a join approach). When building the final output panel, filter to stocks that existed on signal_date — but the return source panel must include delisted history. Removing delisted rows from the daily panel before computing windows introduces survivorship bias.

**Pitfall 7: Applying ESTU moments to all stocks is correct — don't second-guess it.**
Non-ESTU micro-caps get a `mom_score` standardised using ESTU moments. This is intentional: their exposure is measured relative to the investable universe distribution. Do not compute separate standardisation for non-ESTU stocks.

**Pitfall 8: n_obs measures the window, not total history.**
`n_obs` is the count of valid excess-return observations inside the [L+1, T+L] window on signal_date. A stock listed since 1990 with signal_date in 2003 will have n_obs ≤ 504 (capped by T). If Sharadar has gaps in its history for some years, n_obs may be less than 504 even for old stocks. Always count within the window boundaries, not from IPO date.

## §15. Final summary — what "done" looks like

**The deliverable is one file:** `data/out/mom_use4.parquet`

**Acceptance criteria:**

1. ✅ Schema matches §3 exactly: `permaticker` (Int64), `signal_date` (Date), `in_estu` (Bool), `mcap` (Float64), `mom` (Float64), `mom_score` (Float64), `n_obs` (UInt32)
2. ✅ All 6 required validation checks in §12 pass within tolerance
3. ✅ ESTU has ~2,500–3,000 names per date, stable over time (inherited from `estu_monthly.parquet`)
4. ✅ Cap-weighted mean of `mom_score` is machine-zero (|mean| < 1e-6) for every date in ESTU
5. ✅ Equal-weighted std of `mom_score` ≈ 1.0 (|std − 1| < 0.02) for every date in ESTU
6. ✅ `mom` descriptor is a normalised weighted-average daily excess return; post-trim range plausible (|mom| < 0.003 for typical ESTU stocks)
7. ✅ Coverage ≥ 4,000 stocks with non-null `mom_score` per date post-2005
8. ✅ Month-over-month rank stability ρ > 0.85 for ESTU members
9. ✅ Every non-null `mom_score` row has `n_obs ≥ MIN_OBS=252`
10. ✅ All ❓ NOT-IN-PDF decisions in §13 are documented in the code (comment or markdown)

**Once MOM is built and passes validation, the next steps are:**
- MOM is a **standalone** factor with no downstream dependents in the USE4 pipeline.
- Run cross-factor diagnostics: Pearson corr(`mom_score`, `beta_score`) and corr(`mom_score`, `nls_score`) — both should be modest (|corr| < 0.3). High correlation with Beta would suggest the 21-day skip is insufficient or the ESTU benchmark is contaminating the excess return series.
- The remaining USE4 style factors to build: **Residual Volatility (ResVol)** — which depends on Beta — and then Size, Liquidity, EY, B/P, Growth, Leverage, DY.